# 24 - Train Fine-Tashkeel + Inference Pipeline

**Base Model:** `basharalrfooh/Fine-Tashkeel` (ByT5 720M)

**Approach:**
1. Fine-tune Fine-Tashkeel on KSAA training data
2. Use with ASR (seamless_m4t) for final predictions
3. Evaluate on dev set
4. Create submission

**Tasks:**
1. Fine-tune on KSAA data
2. Dev Set: ASR + Fine-tuned Tashkeel + Metrics
3. Blind Test: Create Submission

In [ ]:
# Install dependencies
!pip install -q transformers torch accelerate tqdm datasets sentencepiece pyarabic

In [ ]:
# Setup
import os, sys, json, re, torch, gc, zipfile
from datetime import datetime
from tqdm import tqdm
from pathlib import Path
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
)
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

IS_RUNPOD = False  # Cloud detection removed
PROJECT_DIR = Path('.')
sys.path.insert(0, str(PROJECT_DIR))

# Paths
DATA_DIR = PROJECT_DIR / 'Public_Data_TrainDev'
TRAIN_DIR = DATA_DIR / 'Train'
DEV_INPUT = DATA_DIR / 'dev input-output' / 'Dev_input.json'
DEV_OUTPUT = DATA_DIR / 'dev input-output' / 'Dev_output.json'
TEST_DIR = PROJECT_DIR / 'Test'
TEST_INPUT = TEST_DIR / 'test_input.json'
OUTPUT_DIR = PROJECT_DIR / 'outputs'
SUBMISSION_DIR = PROJECT_DIR / 'submissions'
CHECKPOINTS_DIR = PROJECT_DIR / 'checkpoints' / 'fine_tashkeel_ksaa'

CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
SUBMISSION_DIR.mkdir(exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

def clear_gpu():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()

## Part 1: Tashkeel Training Data

We have audio files in `train_audio/` that can be used to fine-tune SeamlessM4T for Arabic ASR.

### Load ASR Training Data

In [ ]:
# ASR Training Data Preparation
import pandas as pd
from datasets import Dataset, Audio

ASR_TRAIN_SUBSET = 500  # Adjust based on GPU memory

# Load training metadata
train_tsv = TRAIN_DIR / 'train_list.tsv'
train_df = pd.read_csv(train_tsv, sep='\t')
print(f"Total training samples: {len(train_df)}")

# Filter to subset with audio files
audio_dir = TRAIN_DIR / 'train_audio'
train_df['audio_path'] = train_df['stem'].apply(lambda x: audio_dir / f"{x}.mp3")
train_df = train_df[train_df['audio_path'].apply(lambda x: x.exists())]
print(f"Samples with audio: {len(train_df)}")

# Load text for each sample
text_dir = TRAIN_DIR / 'train_text'
train_df['text'] = train_df['stem'].apply(
    lambda x: open(text_dir / f"{x}.txt", 'r', encoding='utf-8').read().strip() if (text_dir / f"{x}.txt").exists() else ""
)

# Remove samples without text
train_df = train_df[train_df['text'].str.len() > 0]
print(f"Samples with text: {len(train_df)}")

# Create dataset - convert stem to string first, then build audio paths
subset_df = train_df[['stem', 'text']].head(ASR_TRAIN_SUBSET).copy()
subset_df['stem'] = subset_df['stem'].astype(str)
subset_df['audio'] = [str(audio_dir / f"{s}.mp3") for s in subset_df['stem']]
asr_data = subset_df[['stem', 'audio', 'text']].to_dict('records')

# Don't cast to Audio - load manually in prepare function
asr_dataset = Dataset.from_list(asr_data)
print(f"ASR dataset: {len(asr_dataset)} samples (will load audio on-the-fly)")

In [ ]:
# Load SeamlessM4T for ASR
from transformers import SeamlessM4TForSpeechToText, SeamlessM4TProcessor

ASR_MODEL_NAME = "facebook/seamlessm4t-medium"

print(f"Loading {ASR_MODEL_NAME}...")
asr_processor = SeamlessM4TProcessor.from_pretrained(ASR_MODEL_NAME)
asr_model = SeamlessM4TForSpeechToText.from_pretrained(
    ASR_MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

print(f"ASR model loaded: {sum(p.numel() for p in asr_model.parameters()):,} params")

In [ ]:
# Prepare SeamlessM4T dataset for training
import librosa

def prepare_asr_dataset(examples):
    # Load audio from path directly
    audio, sr = librosa.load(examples['audio'], sr=16000)
    
    # Process audio using feature extractor (not processor for audio)
    audio_inputs = asr_processor.feature_extractor(audio, sampling_rate=16000, return_tensors="pt")
    input_features = audio_inputs.input_features
    
    # Encode labels using tokenizer separately
    labels = asr_processor.tokenizer(examples['text'], truncation=True, max_length=256).input_ids
    
    return {
        'input_features': input_features.squeeze(0),
        'labels': labels
    }

# Prepare a smaller subset for training
print("Preparing ASR dataset...")
asr_train_dataset = asr_dataset.map(
    prepare_asr_dataset,
    remove_columns=['audio', 'stem'],
    desc="Processing audio"
)

print(f"ASR training dataset: {len(asr_train_dataset)} samples")

In [ ]:
# SeamlessM4T Training Arguments
ASR_CHECKPOINTS_DIR = PROJECT_DIR / 'checkpoints' / 'seamlessm4t_ksaa'
ASR_CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

asr_training_args = Seq2SeqTrainingArguments(
    output_dir=str(ASR_CHECKPOINTS_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=False,
    bf16=True,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    logging_steps=25,
    report_to="none",
    predict_with_generate=True,
    generation_max_length=256,
)

# Use DataCollator
asr_data_collator = DataCollatorForSeq2Seq(
    tokenizer=asr_processor,
    model=asr_model,
    pad_to_multiple_of=8
)

asr_trainer = Seq2SeqTrainer(
    model=asr_model,
    args=asr_training_args,
    train_dataset=asr_train_dataset,
    data_collator=asr_data_collator,
    tokenizer=asr_processor,
)

In [ ]:
# Train ASR Model
clear_gpu()

# Check for existing checkpoint
asr_checkpoints = list(ASR_CHECKPOINTS_DIR.glob('checkpoint-*'))
asr_resume = str(max(asr_checkpoints, key=lambda x: int(x.name.split('-')[1]))) if asr_checkpoints else None

if asr_resume:
    print(f"Resuming ASR training from: {asr_resume}")
    asr_trainer.train(resume_from_checkpoint=asr_resume)
else:
    print("Starting ASR training...")
    asr_trainer.train()

clear_gpu()

# Save ASR model
asr_final_path = ASR_CHECKPOINTS_DIR / 'final'
asr_model.save_pretrained(str(asr_final_path))
asr_tokenizer.save_pretrained(str(asr_final_path))
print(f"ASR model saved to: {asr_final_path}")

In [ ]:
# ASR Inference on Dev/Test Sets
@torch.inference_mode()
def transcribe_audio(audio_path):
    """Transcribe audio using fine-tuned Whisper."""
    audio, sr = librosa.load(audio_path, sr=16000)
    input_features = asr_processor(audio, sampling_rate=16000, return_tensors="pt").input_features
    input_features = input_features.to(device)
    
    forced_decoder_ids = asr_processor.get_decoder_prompt_ids(language="arabic", task="transcribe")
    
    predicted_ids = asr_model.generate(input_features, forced_decoder_ids=forced_decoder_ids)
    transcription = asr_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return transcription

# Generate ASR predictions for dev set
print("Generating ASR predictions for dev set...")
asr_dev_preds = []

for item in tqdm(dev_input, desc="ASR Dev"):
    audio_file = audio_dir / f"{item['id']}.mp3"
    if audio_file.exists():
        try:
            text = transcribe_audio(str(audio_file))
        except:
            text = item['text_undiacritized']
    else:
        text = item['text_undiacritized']
    
    asr_dev_preds.append({'id': item['id'], 'text_diacritized': text})

print(f"Generated {len(asr_dev_preds)} ASR predictions")

# Save ASR predictions
with open(OUTPUT_DIR / 'seamlessm4t_dev_predictions.json', 'w', encoding='utf-8') as f:
    json.dump(asr_dev_preds, f, ensure_ascii=False, indent=2)

In [ ]:
# Generate ASR predictions for test set
print("Generating ASR predictions for test set...")
asr_test_preds = []

for item in tqdm(test_input, desc="ASR Test"):
    audio_file = audio_dir / f"{item['id']}.mp3"
    if audio_file.exists():
        try:
            text = transcribe_audio(str(audio_file))
        except:
            text = item['text_undiacritized']
    else:
        text = item['text_undiacritized']
    
    asr_test_preds.append({'id': item['id'], 'text_diacritized': text})

print(f"Generated {len(asr_test_preds)} ASR test predictions")

# Save ASR test predictions
with open(OUTPUT_DIR / 'seamlessm4t_test_predictions.json', 'w', encoding='utf-8') as f:
    json.dump(asr_test_preds, f, ensure_ascii=False, indent=2)

In [ ]:
# Load KSAA Training Data
import pandas as pd

DIACRITIC_PATTERN = re.compile(r'[\u064B-\u0652]')

train_tsv = TRAIN_DIR / 'train_list.tsv'
train_df = pd.read_csv(train_tsv, sep='\t')
print(f"Training samples: {len(train_df)}")

def load_training_data():
    data = []
    text_dir = TRAIN_DIR / 'train_text'
    
    for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Loading"):
        txt_file = text_dir / f"{row['stem']}.txt"
        if txt_file.exists():
            with open(txt_file, 'r', encoding='utf-8') as f:
                diacritized = f.read().strip()
            undiacritized = DIACRITIC_PATTERN.sub('', diacritized)
            data.append({
                'id': row['stem'],
                'input': undiacritized,
                'target': diacritized
            })
    return data

train_data = load_training_data()
train_data = sorted(train_data, key=lambda x: len(x['input']))
print(f"Loaded {len(train_data)} training samples")

## Part 2: Tashkeel Model

In [ ]:
# Load Fine-Tashkeel
MODEL_NAME = 'basharalrfooh/Fine-Tashkeel'
MODEL_KEY = 'fine_tashkeel_ksaa'

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} params")

In [ ]:
# Prepare dataset
MAX_LENGTH = 512

def preprocess_function(examples):
    inputs = tokenizer(
        examples['input'],
        max_length=MAX_LENGTH,
        truncation=True,
        padding='max_length'
    )
    targets = tokenizer(
        examples['target'],
        max_length=MAX_LENGTH,
        truncation=True,
        padding='max_length'
    )
    inputs['labels'] = targets['input_ids']
    return inputs

# Use subset for faster training
TRAIN_SUBSET = 1000  # Adjust as needed
train_subset = train_data[:TRAIN_SUBSET]
train_dataset = Dataset.from_list(train_subset)
train_dataset = train_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=4,
    remove_columns=['id', 'input', 'target']
)

print(f"Training dataset: {len(train_dataset)} samples")

## Part 3: Tashkeel Training

In [ ]:
# Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINTS_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=3e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=False,  # Disabled due to gradient scaling issues
    bf16=True,   # Use bfloat16 instead (more stable)
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    logging_steps=50,
    report_to="none",
    predict_with_generate=True,
    generation_max_length=MAX_LENGTH,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

In [ ]:
# Train
clear_gpu()

# Use specific checkpoint 200
resume_from = str(CHECKPOINTS_DIR / 'checkpoint-200')

print(f"Resuming from: {resume_from}")

trainer.train(resume_from_checkpoint=resume_from)
clear_gpu()

In [ ]:
# Save final model
final_path = CHECKPOINTS_DIR / 'final'
model.save_pretrained(str(final_path))
tokenizer.save_pretrained(str(final_path))
print(f"Saved to: {final_path}")

## Part 4: Inference Pipeline (ASR + Tashkeel)

In [ ]:
# Load data
with open(DEV_INPUT, 'r', encoding='utf-8') as f:
    dev_input = json.load(f)
with open(DEV_OUTPUT, 'r', encoding='utf-8') as f:
    dev_output = json.load(f)
with open(TEST_INPUT, 'r', encoding='utf-8') as f:
    test_input = json.load(f)

print(f"Dev: {len(dev_input)}, Test: {len(test_input)}")

In [ ]:
# Load fine-tuned model
FINAL_MODEL_PATH = CHECKPOINTS_DIR / 'final'

print("Loading fine-tuned model...")
tashkeel_tokenizer = AutoTokenizer.from_pretrained(str(FINAL_MODEL_PATH))
tashkeel_model = AutoModelForSeq2SeqLM.from_pretrained(
    str(FINAL_MODEL_PATH),
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)
tashkeel_model.eval()
print("Fine-tuned model loaded!")

In [ ]:
# Diacritization function
@torch.inference_mode()
def diacritize_text(text):
    """Add diacritics using fine-tuned model."""
    if not text or not text.strip():
        return text
    
    inputs = tashkeel_tokenizer(text, return_tensors="pt", padding=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    outputs = tashkeel_model.generate(
        **inputs,
        max_length=512,
        num_beams=4,
        early_stopping=True
    )
    
    return tashkeel_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

In [ ]:
# Check for ASR predictions
ASR_DEV = OUTPUT_DIR / 'seamless_m4t_dev_predictions.json'
ASR_TEST = OUTPUT_DIR / 'seamless_m4t_test_predictions.json'

if ASR_DEV.exists():
    with open(ASR_DEV, 'r') as f:
        asr_dev = {p['id']: p['text_diacritized'] for p in json.load(f)}
    print(f"Loaded {len(asr_dev)} ASR dev predictions")
else:
    asr_dev = {}
    print("No ASR dev predictions - using input text")

if ASR_TEST.exists():
    with open(ASR_TEST, 'r') as f:
        asr_test = {p['id']: p['text_diacritized'] for p in json.load(f)}
    print(f"Loaded {len(asr_test)} ASR test predictions")
else:
    asr_test = {}
    print("No ASR test predictions")

In [ ]:
# Metrics - Diac-style evaluation
from diac_evaluation import compute_metrics, print_metrics_table

print("Diac-style evaluation loaded!")

## Dev Set Inference

In [ ]:
# Process dev set
CHECKPOINT_FILE = OUTPUT_DIR / f'{MODEL_KEY}_dev_checkpoint.json'

def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, 'r') as f:
            data = json.load(f)
            return set(data.get('processed_ids', [])), data.get('predictions', [])
    return set(), []

def save_checkpoint(predictions):
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump({
            'processed_ids': [p['id'] for p in predictions],
            'predictions': predictions
        }, f, ensure_ascii=False)

processed_ids, dev_predictions = load_checkpoint()
print(f"Dev: {len(dev_input)} samples, {len(processed_ids)} done")

for item in tqdm(dev_input, desc="Dev"):
    if item['id'] in processed_ids:
        continue
    
    # Get text (from ASR or input)
    if asr_dev and item['id'] in asr_dev:
        text_input = DIACRITIC_PATTERN.sub('', asr_dev[item['id']])
    else:
        text_input = item['text_undiacritized']
    
    # Apply fine-tuned tashkeel
    try:
        diacritized = diacritize_text(text_input)
    except:
        diacritized = text_input
    
    dev_predictions.append({'id': item['id'], 'text_diacritized': diacritized})
    save_checkpoint(dev_predictions)

print(f"Processed {len(dev_predictions)} dev samples")

In [ ]:
# Evaluate
print("="*60)
print("DEV SET RESULTS - Fine-tuned Tashkeel")
print("="*60)

metrics = compute_metrics(dev_predictions, dev_output)
print_metrics_table(metrics, "DEV SET - Fine-tuned Tashkeel")

with open(OUTPUT_DIR / f'{MODEL_KEY}_dev_predictions.json', 'w') as f:
    json.dump(dev_predictions, f, ensure_ascii=False, indent=2)

## Test Set Inference

In [ ]:
# Process test set
TEST_CHECKPOINT = OUTPUT_DIR / f'{MODEL_KEY}_test_checkpoint.json'

def load_test_checkpoint():
    if TEST_CHECKPOINT.exists():
        with open(TEST_CHECKPOINT, 'r') as f:
            data = json.load(f)
            return set(data.get('processed_ids', [])), data.get('predictions', [])
    return set(), []

def save_test_checkpoint(predictions):
    with open(TEST_CHECKPOINT, 'w') as f:
        json.dump({
            'processed_ids': [p['id'] for p in predictions],
            'predictions': predictions
        }, f, ensure_ascii=False)

test_processed, test_predictions = load_test_checkpoint()
print(f"Test: {len(test_input)} samples, {len(test_processed)} done")

for item in tqdm(test_input, desc="Test"):
    if item['id'] in test_processed:
        continue
    
    if asr_test and item['id'] in asr_test:
        text_input = DIACRITIC_PATTERN.sub('', asr_test[item['id']])
    else:
        text_input = item['text_undiacritized']
    
    try:
        diacritized = diacritize_text(text_input)
    except:
        diacritized = text_input
    
    test_predictions.append({'id': item['id'], 'text_diacritized': diacritized})
    save_test_checkpoint(test_predictions)

print(f"Processed {len(test_predictions)} test samples")

In [ ]:
# Create submission
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
json_file = SUBMISSION_DIR / f'{MODEL_KEY}_submission.json'
with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)

zip_file = SUBMISSION_DIR / f'{MODEL_KEY}_submission_{timestamp}.zip'
with zipfile.ZipFile(zip_file, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(json_file, 'submission.json')

print(f"SUBMISSION: {zip_file}")
print(f"Size: {zip_file.stat().st_size/1024:.1f} KB")
print(f"Samples: {len(test_predictions)}")

# Save predictions
with open(OUTPUT_DIR / f'{MODEL_KEY}_test_predictions.json', 'w') as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)
print("Done!")